# Silver | Pesquisas de Satisfação

In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("bronze_table_name", "tb_pesquisas_de_satisfacao")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_pesquisas = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .select(
            "id_pesquisa",
            "id_chamado",
            "nota_atendimento",
            "ingestion_datetime"
        ).distinct()
        .orderBy("id_pesquisa")
)

print(f"Linhas na bronze (sem repetições exatas): {df_pesquisas.count()}")
display(df_pesquisas.limit(5))

# Deduplicação

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_pesquisa").orderBy(F.col("ingestion_datetime").desc())

In [0]:
df_dedup = (
    df_pesquisas
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)


print(f"Total de linhas da tabela de pesquisa de satisfação: {df_dedup.count()}")

# Tratamento

In [0]:
# Transforma ids em string
df_cast = cast_id_to_string(df_dedup)

In [0]:
# Distribuição das notas como vieram da origem
display(df_cast.groupBy("nota_atendimento").count().orderBy("nota_atendimento"))

In [0]:
# "nota_atendimento": conversão para inteiro (vazio e o texto "NULL" da origem viram NULL)
nota_num = F.expr("try_cast(nota_atendimento as int)")

In [0]:
# "nota_atendimento": notas fora do intervalo de 1 a 5 viram NULL
df_limpo = df_cast.withColumn(
    "nota_atendimento",
    F.when(nota_num.between(1, 5), nota_num)
)

display(df_limpo.groupBy("nota_atendimento").count().orderBy("nota_atendimento"))

# Verificações de qualidade

Um mesmo chamado pode ter mais de uma pesquisa registrada (802 chamados com 2 pesquisas), então só `id_pesquisa` precisa ser único. Ao juntar pesquisas com chamados nas camadas seguintes, vale tratar esses casos para não duplicar linhas no cálculo da satisfação.

In [0]:
dq_check_unique("tb_pesquisas_de_satisfacao", "id_pesquisa único", df_limpo, ["id_pesquisa"])

In [0]:
# Chamados com mais de uma pesquisa (esperado: 802)
print(f"Chamados com mais de uma pesquisa: {df_limpo.groupBy('id_chamado').count().filter('count > 1').count()}")

In [0]:
dq_check(
    "tb_pesquisas_de_satisfacao",
    "nota_atendimento deve estar entre 1 e 5 após o tratamento",
    df_limpo,
    F.col("nota_atendimento").isNull() | F.col("nota_atendimento").between(1, 5)
)

In [0]:
# Pesquisas devem apontar para chamados que existem em tb_chamados
df_ids_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
        .select("id_chamado")
        .distinct()
        .withColumn("chamado_existe", F.lit(True))
)

dq_check(
    "tb_pesquisas_de_satisfacao",
    "id_chamado deve existir em tb_chamados",
    df_limpo.join(df_ids_chamados, "id_chamado", "left"),
    F.col("chamado_existe").isNotNull()
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_pesquisas_de_satisfacao")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_limpo, silver_table_name)